In [2]:
import os
import numpy as np
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Ścieżka do danych
data_dir = 'asl_alphabet_train'
categories = sorted(os.listdir(data_dir))  # np. ['A', 'B', ..., 'nothing']
img_size = 64  # można użyć 64x64 zamiast 200x200 dla mniejszego modelu

# Wczytaj dane
data = []
labels = []

for idx, category in enumerate(categories):
    path = os.path.join(data_dir, category)
    for img_name in tqdm(os.listdir(path)[:1000]):  # dla szybkości: max 1000 obrazów na kategorię
        try:
            img_path = os.path.join(path, img_name)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (img_size, img_size))
            data.append(img)
            labels.append(idx)
        except Exception as e:
            print(f"Problem z {img_path}: {e}")

data = np.array(data) / 255.0  # normalizacja
labels = to_categorical(labels)

# Podział na zbiory
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

# Budowa modelu CNN
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(categories), activation='softmax')  # 29 klas
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Trenowanie
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

# Zapis modelu
model.save('asl_model_v1.keras')
print("Model zapisany jako 'asl_model_v1.keras'")

100%|██████████| 1000/1000 [00:00<00:00, 1547.39it/s]
C:\6 SEM\IS\lab2\lab66666\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,628,893 (6.21 MB)

 Trainable params: 1,628,893 (6.21 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 36s 47ms/step - accuracy: 0.2010 - loss: 2.7288 - val_accuracy: 0.7967 - val_loss: 0.7683
Epoch 2/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.6626 - loss: 0.9715 - val_accuracy: 0.9319 - val_loss: 0.3199
Epoch 3/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.7925 - loss: 0.5726 - val_accuracy: 0.9652 - val_loss: 0.1848
Epoch 4/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.8462 - loss: 0.4260 - val_accuracy: 0.9710 - val_loss: 0.1229
Epoch 5/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.8800 - loss: 0.3260 - val_accuracy: 0.9769 - val_loss: 0.1006
Epoch 6/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 31s 42ms/step - accuracy: 0.9017 - loss: 0.2706 - val_accuracy: 0.9793 - val_loss: 0.0833
Epoch 7/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 31s 42ms/step - accuracy: 0.9108 - loss: 0.2486 - val_accuracy: 0.9840 - val_loss: 0.0624
Epoch 8/10
725/725 ━━━━━━━━━━━━━━━━━━━━ 30s 41ms/step - accuracy: 0.9167 - loss: 0.2287 - 

In [3]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import os

# wczytaj model
model = load_model('asl_model_v1.keras')

# załaduj klasy
classes = sorted(os.listdir("asl_alphabet_train"))
print("Załadowane klasy:", classes)

# inicjalizacja kamery
cap = cv2.VideoCapture(0)
predicted_text = ""
current_letter = ""

while True:
    ret, frame = cap.read()
    if not ret:
        break

    roi = frame[100:400, 100:400]
    roi_resized = cv2.resize(roi, (64, 64))
    roi_normalized = roi_resized / 255.0
    roi_expanded = np.expand_dims(roi_normalized, axis=0)

    prediction = model.predict(roi_expanded, verbose=0)
    predicted_class = np.argmax(prediction)
    predicted_letter = classes[predicted_class]
    current_letter = predicted_letter

    # Wyświetl dane
    cv2.rectangle(frame, (100, 100), (400, 400), (0, 255, 0), 2)
    cv2.putText(frame, f'Aktualna litera: {current_letter}', (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.putText(frame, f'Tekst: {predicted_text}', (10, 450),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    cv2.imshow("ASL Recognition", frame)

# trzeba wcisnąć s, aby dodać tę literę do napisu;  c, aby wyczyścić napis; q, aby zakończyć
    
    key = cv2.waitKey(1)
    if key == ord('q'):
        break
    elif key == ord('s'):
        if current_letter != "nothing":
            predicted_text += current_letter
    elif key == ord('c'):
        predicted_text = ""

cap.release()
cv2.destroyAllWindows()

Załadowane klasy: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
